In [1]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
FIGURES_DIR = PROJECT_DIR / "results" / "figures"

# Create figures directory if needed
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load metadata
df = pd.read_csv(DATA_DIR / "master_dataset.csv")

print(f"Articles: {len(df)}")
print(f"Countries: {df['country'].nunique()}")
print()

print("Articles by country:")
print(df["country"].value_counts())

print()
print("Columns:")
print(df.columns.tolist())

Articles: 36
Countries: 3

Articles by country:
country
Australia         17
British Malaya    10
India              9
Name: count, dtype: int64

Columns:
['article_id', 'country', 'newspaper', 'date', 'page', 'title', 'theme', 'sub_theme', 'population', 'author', 'source_url', 'archive', 'ocr_quality']


In [2]:
import re
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Load article text
articles = []

for _, row in df.iterrows():

    country_folder = row["country"].lower().replace(" ", "-")

    # British Malaya folder is already named british-malaya
    text_path = RAW_DIR / country_folder / f"{row['article_id']}.txt"

    text = text_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).strip()

    articles.append({
        "article_id": row["article_id"],
        "country": row["country"],
        "text": text
    })

analysis_df = pd.DataFrame(articles)


# Conservative OCR cleaning used in the preprocessing notebook
def clean_ocr_spacing(text):
    cleaned = text

    # Normalize line endings
    cleaned = cleaned.replace("\r\n", "\n").replace("\r", "\n")

    # Remove known OCR marker
    cleaned = cleaned.replace("★", " ")

    # Normalize curly quotation marks
    cleaned = (
        cleaned.replace("“", '"')
               .replace("”", '"')
               .replace("‘", "'")
               .replace("’", "'")
    )

    # Fix sentence-ending punctuation followed immediately
    # by a capital letter
    cleaned = re.sub(r'([.!?])([A-Z])', r'\1 \2', cleaned)

    # Fix punctuation + closing quote + capital letter
    cleaned = re.sub(
        r'([.!?]["\'])([A-Z])',
        r'\1 \2',
        cleaned
    )

    # Confirmed OCR spacing corrections
    cleaned = re.sub(
        r'1939malaya',
        '1939 Malaya',
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r'chellappahdistinguished',
        'chellappah distinguished',
        cleaned,
        flags=re.IGNORECASE
    )

    # Collapse repeated horizontal whitespace
    cleaned = re.sub(r'[ \t]+', ' ', cleaned)

    # Preserve paragraph structure
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)

    return cleaned.strip()


analysis_df["text"] = analysis_df["text"].apply(
    clean_ocr_spacing
)


# Tokenization
analysis_df["tokens"] = analysis_df["text"].apply(
    word_tokenize
)

# Keep alphabetic word tokens
analysis_df["word_tokens"] = analysis_df["tokens"].apply(
    lambda tokens: [
        token.lower()
        for token in tokens
        if any(char.isalpha() for char in token)
    ]
)

# English stopwords
english_stopwords = set(
    stopwords.words("english")
)

analysis_df["filtered_tokens"] = analysis_df["word_tokens"].apply(
    lambda tokens: [
        token
        for token in tokens
        if token not in english_stopwords
    ]
)

# Remove identified OCR/metadata artefacts
artifact_tokens = {
    "mr.", "mrs.", "mrs",
    "l.", "m.", "a.", "d.", "s.",
    "'s"
}

analysis_df["clean_tokens"] = analysis_df[
    "filtered_tokens"
].apply(
    lambda tokens: [
        token
        for token in tokens
        if token not in artifact_tokens
    ]
)


# Basic verification
clean_words = [
    word
    for tokens in analysis_df["clean_tokens"]
    for word in tokens
]

print(f"Articles loaded: {len(analysis_df)}")
print(f"Clean tokens: {len(clean_words):,}")
print(f"Unique clean word types: {len(set(clean_words)):,}")

Articles loaded: 36
Clean tokens: 8,380
Unique clean word types: 3,625


In [3]:
# Corpus composition by country

corpus_table = (
    analysis_df
    .groupby("country")
    .agg(
        articles=("article_id", "count"),
        clean_tokens=("clean_tokens", "sum")
    )
    .reindex(["India", "British Malaya", "Australia"])
)

# Convert token lists to their lengths
corpus_table["clean_tokens"] = (
    analysis_df
    .groupby("country")["clean_tokens"]
    .apply(lambda x: sum(len(tokens) for tokens in x))
    .reindex(["India", "British Malaya", "Australia"])
)

corpus_table["percentage_of_articles"] = (
    corpus_table["articles"]
    / len(analysis_df)
    * 100
)

print(corpus_table.round(1))

                articles  clean_tokens  percentage_of_articles
country                                                       
India                  9          2302                    25.0
British Malaya        10          3178                    27.8
Australia             17          2900                    47.2


In [4]:
# Thematic distribution of sampled articles

theme_rows = []

for country in ["India", "British Malaya", "Australia"]:

    country_df = df[df["country"] == country]
    total_articles = len(country_df)

    for theme in ["People", "Land", "Culture"]:

        count = country_df["theme"].str.contains(
            theme,
            regex=False,
            na=False
        ).sum()

        percentage = (count / total_articles) * 100

        theme_rows.append({
            "country": country,
            "theme": theme,
            "articles": count,
            "percentage": percentage
        })

theme_table = pd.DataFrame(theme_rows)

# Display in a compact format
theme_table["percentage"] = theme_table["percentage"].round(1)

print(theme_table.to_string(index=False))

       country   theme  articles  percentage
         India  People         9       100.0
         India    Land         1        11.1
         India Culture         2        22.2
British Malaya  People        10       100.0
British Malaya    Land         3        30.0
British Malaya Culture         1        10.0
     Australia  People        15        88.2
     Australia    Land         9        52.9
     Australia Culture         6        35.3


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create one document per article from the cleaned tokens
documents = analysis_df["clean_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

# Article-level TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=False,
    min_df=2,
    max_df=0.80,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(documents)

terms = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=terms,
    index=analysis_df["article_id"]
)

# Attach country information
tfidf_with_country = tfidf_df.copy()
tfidf_with_country["country"] = (
    analysis_df.set_index("article_id")["country"]
)

# Mean article-level TF-IDF by country
country_tfidf = (
    tfidf_with_country
    .groupby("country")
    .mean(numeric_only=True)
)

# Create final table
tfidf_rows = []

for country in ["India", "British Malaya", "Australia"]:

    top_terms = (
        country_tfidf.loc[country]
        .sort_values(ascending=False)
        .head(10)
    )

    for rank, (term, score) in enumerate(
        top_terms.items(),
        start=1
    ):
        tfidf_rows.append({
            "country": country,
            "rank": rank,
            "term": term,
            "mean_tfidf": score
        })

tfidf_table = pd.DataFrame(tfidf_rows)

tfidf_table["mean_tfidf"] = (
    tfidf_table["mean_tfidf"].round(4)
)

print(tfidf_table.to_string(index=False))

       country  rank       term  mean_tfidf
         India     1   congress      0.0911
         India     2     muslim      0.0677
         India     3 conference      0.0645
         India     4      india      0.0605
         India     5 government      0.0583
         India     6     united      0.0536
         India     7   ministry      0.0534
         India     8   district      0.0472
         India     9    freedom      0.0459
         India    10      could      0.0457
British Malaya     1     malaya      0.0919
British Malaya     2    indians      0.0797
British Malaya     3      malay      0.0718
British Malaya     4      india      0.0698
British Malaya     5      times      0.0670
British Malaya     6    british      0.0660
British Malaya     7    chinese      0.0659
British Malaya     8     malays      0.0640
British Malaya     9        sir      0.0564
British Malaya    10    malayan      0.0555
     Australia     1 aborigines      0.1228
     Australia     2      would 

In [6]:
from collections import Counter

frequency_rows = []

for country in ["India", "British Malaya", "Australia"]:

    country_df = analysis_df[
        analysis_df["country"] == country
    ]

    counts = Counter()

    for tokens in country_df["clean_tokens"]:
        counts.update(tokens)

    total_tokens = sum(counts.values())

    for rank, (term, count) in enumerate(
        counts.most_common(20),
        start=1
    ):
        frequency_rows.append({
            "country": country,
            "rank": rank,
            "term": term,
            "count": count,
            "per_1000_tokens": (count / total_tokens) * 1000
        })

frequency_table = pd.DataFrame(frequency_rows)

frequency_table["per_1000_tokens"] = (
    frequency_table["per_1000_tokens"].round(2)
)

print(frequency_table.to_string(index=False))

       country  rank       term  count  per_1000_tokens
         India     1   congress     24            10.43
         India     2     tribes     20             8.69
         India     3      india     18             7.82
         India     4 government     15             6.52
         India     5 aboriginal     12             5.21
         India     6     people     12             5.21
         India     7 conference     12             5.21
         India     8 resolution     10             4.34
         India     9      areas     10             4.34
         India    10        may     10             4.34
         India    11       must     10             4.34
         India    12   ministry      9             3.91
         India    13   district      9             3.91
         India    14       said      9             3.91
         India    15    freedom      9             3.91
         India    16      world      9             3.91
         India    17    message      9          

In [7]:
# Save final results tables

corpus_table.to_csv(
    FIGURES_DIR.parent / "corpus_composition.csv"
)

theme_table.to_csv(
    FIGURES_DIR.parent / "thematic_distribution.csv",
    index=False
)

tfidf_table.to_csv(
    FIGURES_DIR.parent / "article_level_tfidf.csv",
    index=False
)

frequency_table.to_csv(
    FIGURES_DIR.parent / "normalized_frequencies.csv",
    index=False
)

print("Saved final results tables to:")
print(FIGURES_DIR.parent)

Saved final results tables to:
c:\Users\User\Desktop\colonial-discourse-nlp\results
